In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

raw_path = Path("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df_features = pd.read_csv(raw_path)

# Reproduce the documented TotalCharges cleaning decision
total_charges_text = df_features["TotalCharges"].str.strip().replace("", np.nan)
blank_total_charges = total_charges_text.isna()

# Guard against applying the zero-value assumption to the wrong records
assert (df_features.loc[blank_total_charges, "tenure"] == 0).all()

df_features["TotalCharges"] = (
    pd.to_numeric(total_charges_text, errors="raise")
    .fillna(0)
)

# Numeric version of the target for rates and later analysis
df_features["ChurnFlag"] = (df_features["Churn"] == "Yes").astype(int)

print(df_features.shape)
print(df_features["TotalCharges"].dtype)
print("Missing TotalCharges:", df_features["TotalCharges"].isna().sum())

(7043, 22)
float64
Missing TotalCharges: 0


In [3]:
# Binary behavioral features
df_features["HasInternet"] = (
    df_features["InternetService"] != "No"
).astype(int)

df_features["HasPhone"] = (
    df_features["PhoneService"] == "Yes"
).astype(int)

df_features["AutoPayment"] = (
    df_features["PaymentMethod"]
    .isin(["Bank transfer (automatic)", "Credit card (automatic)"])
).astype(int)

# Count subscribed add-on services.
# “No internet service” and “No phone service” are not treated as subscribed services.
add_on_columns = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
]

df_features["ServiceCount"] = (
    df_features[add_on_columns]
    .eq("Yes")
    .sum(axis=1)
)

# Tenure bands make lifecycle stages easier to communicate in EDA and segmentation.
df_features["TenureGroup"] = pd.cut(
    df_features["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0–12 months", "13–24 months", "25–48 months", "49–72 months"]
)

# Validation
display(
    df_features[
        ["tenure", "TenureGroup", "MonthlyCharges", "ServiceCount",
         "HasInternet", "HasPhone", "AutoPayment", "Churn"]
    ].head()
)

print("Missing values after feature engineering:")
display(df_features.isna().sum()[df_features.isna().sum() > 0])

,tenure,TenureGroup,MonthlyCharges,ServiceCount,HasInternet,HasPhone,AutoPayment,Churn
0,1,0–12 months,29.85,1,1,0,0,No
1,34,25–48 months,56.95,2,1,1,0,No
2,2,0–12 months,53.85,2,1,1,0,Yes
3,45,25–48 months,42.30,3,1,0,1,No
4,2,0–12 months,70.70,0,1,1,0,Yes


Missing values after feature engineering:


Series([], dtype: int64)

In [4]:
(
    df_features.groupby("TenureGroup", observed=True)
    .agg(
        customers=("customerID", "count"),
        churn_rate=("ChurnFlag", "mean"),
        average_monthly_charges=("MonthlyCharges", "mean")
    )
    .assign(churn_rate=lambda x: x["churn_rate"].map("{:.2%}".format))
)

,customers,churn_rate,average_monthly_charges
TenureGroup,,,
0–12 months,2186,47.44%,56.097781
13–24 months,1024,28.71%,61.357275
25–48 months,1594,20.39%,65.930552
49–72 months,2239,9.51%,73.945377


## Customer lifecycle finding

Churn declines substantially as customer tenure increases. The first 12 months represent the most vulnerable lifecycle stage: 47.44% of customers in this group churned, compared with 9.51% of customers with 49–72 months of tenure.

This suggests that onboarding, early service experience, and first-year retention offers should be investigated as priority areas. This descriptive result does not establish causation.

In [10]:
from pathlib import Path

processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

df_features.to_csv(
    processed_path / "telco_churn_features.csv",
    index=False
)